# Orca Nano — QLoRA Fine-Tune v4 (Colab free T4)

Run cells top to bottom. Before starting: **Runtime → Change runtime type → T4 GPU**.

**Why back on Colab, not Kaggle**: Kaggle's session died silently multiple
times today, losing a fully-trained model each time with no checkpoint to
recover from — a worse failure mode than anything Colab hit (a GPU quota
wall, one download stall, both one-off and worked around). Colab's disk is
also much larger (~80-100GB vs Kaggle's 19.5GB working-directory quota),
so the GGUF disk-space crash from Kaggle shouldn't happen here at all.

**Every fix learned today, baked in from the start (not patched in after
a crash):**
- Plain PyPI `unsloth` install — no `git+https://...` source, which
  triggers a slow git-clone + build-from-source step every run.
- Xet fast-download path disabled before any import (stalled a model
  download once, mid-session, without this).
- `average_tokens_across_devices=False` in `TrainingArguments` — without
  it, training crashes partway with `AttributeError: 'int' object has no
  attribute 'mean'` (a known Unsloth/Transformers incompatibility,
  [unslothai/unsloth#3769](https://github.com/unslothai/unsloth/issues/3769)).
- Dynamic uploaded-filename detection — handles Colab's `(1)`/`(2)`
  suffix automatically if a same-named file already exists in the session.
- **Export straight to Google Drive from the start** — `files.download()`
  failed 3 times in a row on v1/v2 from page reloads killing the transfer
  mid-download. Drive-copy is what actually worked reliably that time.

Same training config throughout: rank 16 LoRA, 2050/108 dataset (full nano
distillation batch + 150 targeted `honesty_hedging` examples), 2 epochs
(not 3 — v1's validation loss rose after step 100, a real overfitting
signal on this dataset size that a later, healthier run confirmed: at
2 epochs validation loss stayed *below* training loss the whole way).

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"

# Plain PyPI install — no git+https source, no build-from-source step.
!pip install -q unsloth trl transformers datasets peft bitsandbytes accelerate

## Upload your training data

Upload the two files from your Desktop: `orca_llama3_train.jsonl` and `orca_llama3_eval.jsonl`
(2050 train / 108 eval examples).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select orca_llama3_train.jsonl and orca_llama3_eval.jsonl from Desktop
print('Uploaded:', list(uploaded.keys()))

In [ ]:
import json

def load_jsonl(path):
    lines = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    lines.append(json.loads(line))
                except Exception:
                    pass
    return lines

# Find uploaded filenames dynamically instead of assuming exact names — Colab
# appends ' (1)', ' (2)', etc. if a same-named file already exists on this
# runtime.
train_filename = next((f for f in uploaded if 'train' in f.lower()), None)
eval_filename  = next((f for f in uploaded if 'eval' in f.lower()), None)

if train_filename is None:
    raise FileNotFoundError("No training file found in uploaded files. Please re-run the upload cell above.")

raw_train = load_jsonl(train_filename)
raw_eval  = load_jsonl(eval_filename) if eval_filename else raw_train[:max(1, len(raw_train)//10)]
print(f'train={len(raw_train)} eval={len(raw_eval)}')

## Load base model (4-bit) + attach LoRA

Rank 16 — sized for a free-tier GPU's VRAM, not the 128-rank A100 cloud preset.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
base_model = "unsloth/Qwen2.5-7B-Instruct"  # exact case matters on Hugging Face

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import Dataset

def format_conv(ex):
    turns = ex.get("conversations", ex.get("text"))
    if isinstance(turns, str):
        return turns  # already-formatted llama3 text field
    parts = []
    for t in turns:
        role = t.get("role", "")
        val  = t.get("value", "")
        if role == "system":
            parts.append(f"<|start_header_id|>system<|end_header_id|>\n\n{val}<|eot_id|>")
        elif role == "human":
            parts.append(f"<|start_header_id|>user<|end_header_id|>\n\n{val}<|eot_id|>")
        elif role == "gpt":
            parts.append(f"<|start_header_id|>assistant<|end_header_id|>\n\n{val}<|eot_id|>")
    return "".join(parts)

# The formatter.py output already has a 'text' field per example (llama3 format) — use it directly if present.
train_ds = Dataset.from_list([{"text": ex["text"] if "text" in ex else format_conv(ex)} for ex in raw_train])
eval_ds  = Dataset.from_list([{"text": ex["text"] if "text" in ex else format_conv(ex)} for ex in raw_eval])
print(f'train_ds={len(train_ds)} eval_ds={len(eval_ds)}')

## Train

Batch size 2 + grad accumulation 4 (effective batch 8), 2 epochs.
`average_tokens_across_devices=False` prevents the known Unsloth/Transformers
crash — see the intro cell for why.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import time

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_steps=100,
        save_strategy="no",
        output_dir="output",
        eval_strategy="steps",
        report_to="none",
        average_tokens_across_devices=False,
    ),
)

print("[train] starting QLoRA training...")
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"[train] done in {elapsed:.1f} min")

## Merge LoRA + export GGUF

In [ ]:
print("[merge] merging LoRA adapters...")
model.save_pretrained_merged("merged", tokenizer, save_method="merged_16bit")
print("[merge] saved to ./merged")

print("[gguf] converting to GGUF q4_k_m...")
model.save_pretrained_gguf("gguf", tokenizer, quantization_method="q4_k_m")
print("[gguf] saved to ./gguf")

## Copy to Google Drive immediately, then download from drive.google.com

Not using `files.download()` at all this time — it needs a continuously
live browser tab connection and failed repeatedly on earlier rounds from
page reloads. Copying to Drive right after conversion, before anything
else can go wrong, is the one export method that's actually worked
reliably so far.

In [ ]:
import glob
import shutil
from google.colab import drive

drive.mount('/content/drive')

# Recursive + case-insensitive search — unsloth's output folder naming has
# varied between runs (e.g. 'gguf/' vs 'gguf_gguf/'), so search broadly
# instead of assuming one exact path.
candidates = [f for f in glob.glob('**/*.gguf', recursive=True) if 'q4_k_m' in f.lower()]
print('Found:', candidates)

if candidates:
    source_path = candidates[0]
    filename = source_path.split('/')[-1]
    dest_path = f'/content/drive/MyDrive/{filename}'
    print(f'Copying {filename} to Google Drive...')
    shutil.copy(source_path, dest_path)
    print(f'Done! File saved to: {dest_path}')
    print('Now go to drive.google.com and download it from the root of My Drive.')
else:
    print('No GGUF file found — check the [gguf] step above for errors.')